# Retos del workshop RAG · notebook de arranque

Este notebook trae el punto de partida de algunos retos. Los bloques marcados con `TODO` son tuyos. Copia este archivo a `notebooks/04_challenges.ipynb` en tu repo.

## C1 · HNSW contra fuerza bruta con FAISS

Instala FAISS y ejecuta la línea base. Después completa los TODO.

In [ ]:
%pip install -q faiss-cpu matplotlib

In [ ]:
import time
import faiss
import numpy as np

d, n, nq, k = 64, 20_000, 200, 10
rng = np.random.default_rng(42)
xb = rng.random((n, d), dtype="float32")
xq = rng.random((nq, d), dtype="float32")

flat = faiss.IndexFlatL2(d)
flat.add(xb)
t0 = time.time(); _, I_true = flat.search(xq, k); t_flat = (time.time() - t0) / nq * 1000

def recall_at_k(I, I_true):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(I, I_true)]))

rows = []
for M in (8, 16, 32):
    index = faiss.IndexHNSWFlat(d, M)
    index.hnsw.efConstruction = 100
    t0 = time.time(); index.add(xb); build_s = time.time() - t0
    for ef in (10, 16, 32, 64, 128, 256):
        index.hnsw.efSearch = ef
        t0 = time.time(); _, I = index.search(xq, k); ms = (time.time() - t0) / nq * 1000
        rows.append((M, ef, recall_at_k(I, I_true), ms, build_s))

print(f"exacto: {t_flat:.3f} ms/consulta")
for M, ef, rec, ms, b in rows:
    print(f"M={M:>2} ef={ef:>3} recall@{k}={rec:.3f}  {ms:.3f} ms/q  (build {b:.1f}s)")

In [ ]:
# TODO 1: grafica recall vs efSearch (una curva por M) con matplotlib.
# TODO 2: repite el experimento con n = 100_000 y n = 500_000. ¿Desde qué n HNSW supera al exacto?
# TODO 3: escribe tus conclusiones en RESPUESTAS.md

## C4 · Evaluación automática de la recuperación

Necesitas las variables del notebook 03 (`retriever`, `all_docs`, …) o reconstruirlas aquí.

In [ ]:
import csv

def cargar_preguntas(ruta="eval/preguntas.csv"):
    with open(ruta, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def hit_at_k(retriever, filas):
    hits = 0
    for r in filas:
        docs = retriever.invoke(r["pregunta"])
        ok = any(
            r["palabra_clave"].lower() in d.page_content.lower()
            and d.metadata["source"] == r["fuente_esperada"]
            for d in docs
        )
        hits += ok
    return hits / len(filas)

# TODO: amplía eval/preguntas.csv a 10 filas, calcula hit@3 y hit@5 con chunk_size 400, 900 y 1500.

## C6 · Persistencia del índice en disco

Punto de partida: cambia el cliente en memoria por uno con ruta.

In [ ]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(path="./qdrant_data")
existe = qdrant_client.collection_exists("academic_papers_nvidia")
print("¿Ya existe la colección?", existe)

# TODO: si no existe, indexa y mide el tiempo. Si existe, reutilízala y mide el tiempo.
# TODO: reinicia el runtime y comprueba que sigue respondiendo sin reindexar.